# Build and publish reusable ASL cache

Chạy notebook này một lần bằng Colab CLI. Nó download raw archive, audit, MediaPipe crop, tạo Parquet/manifest và publish derived cache lên cùng Hugging Face Dataset. Cần biến môi trường `HF_TOKEN` có quyền write.

In [ ]:
%pip -q install 'mediapipe>=0.10.14' 'huggingface-hub>=0.25' pandas pyarrow
# Colab đã có TensorFlow; notebook cache không cần cài lại TensorFlow.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, urllib.request, zipfile
import cv2, mediapipe as mp, pandas as pd
from huggingface_hub import HfApi, snapshot_download

REPO_ID = 'hnam25/asl-hand-gesture-images'
RAW_ARCHIVE = 'ASL_HG_36000/ASL_Raw_Images.zip'
CACHE_PREFIX = 'derived/mediapipe-hand-landmarker-v1'
ROOT = Path('/content/asl-cache-build') if Path('/content').exists() else Path.cwd() / 'asl-cache-build'
RAW, PROCESSED, META = ROOT/'data/raw', ROOT/'data/processed', ROOT/'data/metadata'
for directory in (RAW, PROCESSED, META, ROOT/'assets'): directory.mkdir(parents=True, exist_ok=True)
CLASSES = [str(i) for i in range(10)] + [chr(i) for i in range(ord('A'), ord('Z') + 1)]
PADDING_RATIO = .18

def log(stage, **fields):
    message = f"{datetime.now(timezone.utc).isoformat()} | {stage} | {json.dumps(fields, ensure_ascii=False)}"
    print(message, flush=True)

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''): digest.update(chunk)
    return digest.hexdigest()

snapshot_download(repo_id=REPO_ID, repo_type='dataset', local_dir=RAW, allow_patterns=RAW_ARCHIVE)
raw_zip = next(RAW.rglob('ASL_Raw_Images.zip'))
RAW_ARCHIVE_SHA256 = sha256_file(raw_zip)
log('raw_archive_ready', archive=str(raw_zip), sha256=RAW_ARCHIVE_SHA256)
with zipfile.ZipFile(raw_zip) as archive:
    archive.extractall(RAW)
roots = [path.parent for path in RAW.rglob('0') if path.is_dir() and all((path.parent / label).is_dir() for label in CLASSES)]
if not roots: raise RuntimeError('Could not find class root 0-9/A-Z after raw archive extraction.')
CLASS_ROOT = sorted(roots, key=lambda path: len(path.parts))[0]
log('class_root_ready', class_root=str(CLASS_ROOT))

In [ ]:
# Audit source files and write portable relative paths into Parquet.
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
rows = []
for label in CLASSES:
    for path in sorted((CLASS_ROOT / label).rglob('*')):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS: continue
        image = cv2.imread(str(path))
        rows.append({'label': label, 'source_relative': str(path.relative_to(CLASS_ROOT)), 'status': 'ok' if image is not None else 'unreadable', 'sha256': sha256_file(path) if image is not None else ''})
audit = pd.DataFrame(rows)
audit.to_parquet(META/'audit.parquet', index=False)
duplicates = audit[(audit.sha256 != '') & audit.sha256.duplicated(False)].copy()
duplicates.to_parquet(META/'duplicates.parquet', index=False)
counts = audit[audit.status == 'ok'].groupby('label').size().reindex(CLASSES, fill_value=0)
if counts.min() < 10: raise RuntimeError(f'Invalid class distribution: {counts.to_dict()}')
log('audit_complete', readable=int((audit.status == 'ok').sum()), unreadable=int((audit.status != 'ok').sum()), duplicates=len(duplicates), per_class=counts.to_dict())

In [ ]:
# Crop hands exactly once; manifest stores relative paths so it is portable across Colab sessions.
TASK_MODEL = ROOT/'assets/hand_landmarker.task'
if not TASK_MODEL.exists(): urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task', TASK_MODEL)
options = mp.tasks.vision.HandLandmarkerOptions(base_options=mp.tasks.BaseOptions(model_asset_path=str(TASK_MODEL)), running_mode=mp.tasks.vision.RunningMode.IMAGE, num_hands=1, min_hand_detection_confidence=.5, min_hand_presence_confidence=.5, min_tracking_confidence=.5)
landmarker = mp.tasks.vision.HandLandmarker.create_from_options(options)
results = []
try:
    for index, row in enumerate(audit[audit.status == 'ok'].itertuples(), 1):
        image = cv2.imread(str(CLASS_ROOT / row.source_relative)); rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        detected = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
        destination_relative, status = row.source_relative, 'ok'
        if not detected.hand_landmarks:
            status = 'no_hand_detected'
        else:
            height, width = image.shape[:2]; points = detected.hand_landmarks[0]
            xs, ys = [point.x*width for point in points], [point.y*height for point in points]
            side = max(max(xs)-min(xs), max(ys)-min(ys)) * (1 + 2 * PADDING_RATIO); cx, cy = (min(xs)+max(xs))/2, (min(ys)+max(ys))/2
            x1, y1, x2, y2 = max(0,int(cx-side/2)), max(0,int(cy-side/2)), min(width,int(cx+side/2)), min(height,int(cy+side/2))
            if x2 <= x1 or y2 <= y1: status = 'invalid_bbox'
            else:
                output = PROCESSED/destination_relative; output.parent.mkdir(parents=True, exist_ok=True); cv2.imwrite(str(output), image[y1:y2, x1:x2])
        results.append({'label': row.label, 'source_relative': row.source_relative, 'processed_relative': destination_relative, 'status': status})
        if index % 1000 == 0: log('segmentation_progress', processed=index)
finally:
    landmarker.close()
segmentation = pd.DataFrame(results); segmentation.to_parquet(META/'segmentation_manifest.parquet', index=False)
log('segmentation_complete', results=segmentation.status.value_counts().to_dict())

In [ ]:
# Create one compressed crop archive and publish cache artifacts to the same Hugging Face Dataset.
token = os.getenv('HF_TOKEN')
if not token: raise RuntimeError('Set HF_TOKEN with write access before publishing, e.g. export HF_TOKEN=hf_...')
processed_zip = Path(shutil.make_archive(str(META/'processed_hand_crops'), 'zip', root_dir=PROCESSED))
manifest = {'cache_schema_version': 1, 'dataset_repo': REPO_ID, 'raw_archive': RAW_ARCHIVE, 'raw_archive_sha256': RAW_ARCHIVE_SHA256, 'class_count': len(CLASSES), 'image_count': int(len(audit)), 'segmentation_config': {'model_url': 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task', 'padding_ratio': PADDING_RATIO, 'min_detection_confidence': .5, 'min_presence_confidence': .5, 'min_tracking_confidence': .5}, 'created_at_utc': datetime.now(timezone.utc).isoformat()}
manifest_path = META/'cache_manifest.json'; manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
api = HfApi(token=token)
for path in (META/'audit.parquet', META/'duplicates.parquet', META/'segmentation_manifest.parquet', manifest_path, processed_zip):
    target = f'{CACHE_PREFIX}/{path.name}'
    log('upload_start', file=path.name, bytes=path.stat().st_size)
    api.upload_file(path_or_fileobj=str(path), path_in_repo=target, repo_id=REPO_ID, repo_type='dataset', commit_message=f'Publish {CACHE_PREFIX} cache')
log('publish_complete', cache_prefix=CACHE_PREFIX)
print(f'Published reusable cache to https://huggingface.co/datasets/{REPO_ID}/tree/main/{CACHE_PREFIX}')

## Next run

The end-to-end notebook verifies the SHA-256 of the downloaded raw archive against `cache_manifest.json`. It can safely reuse this cache only when fingerprint and MediaPipe configuration match.